In [72]:
import requests
import json
import pandas as pd
import geopandas as gpd
import os
import re
import plotly.express as px
from collections import defaultdict
from geojson import Feature, FeatureCollection
from thefuzz import fuzz, process
from shapely.geometry import mapping

In [200]:
pd.set_option('display.max_rows', 500)

In [561]:
STATE_GEO_VALUES = {
    "Baden-Württemberg": (48.7, 9.1, 6.9),
    "Bayern": (48.99, 11.45, 6.4),
    "Brandenburg": (52.45, 13.1, 6.8),
    "Bremen": (53.0793, 8.8017, 8),  # adjust for Bremerhaven
    "Hamburg": (53.55, 10, 9),
    "Hessen": (50.6521, 9.1624, 8),
    "Mecklenburg-Vorpommern": (53.6127, 12.4296, 8),
    "Niedersachsen": (52.6367, 9.8451, 8),
    "Nordrhein-Westfalen": (51.4332, 7.6616, 7),
    "Rheinland-Pfalz": (50.1183, 7.3090, 8),
    "Saarland": (49.3964, 7.0230, 8),
    "Sachsen": (51.1045, 13.2017, 8),
    "Sachsen-Anhalt": (51.9503, 11.6923, 8),
    "Schleswig-Holstein": (54.5250, 9.5608, 8),
    "Thüringen": (51.0110, 10.8453, 8),
    "(Deutschland)": (51.1657, 10.4515, 5)
}

In [332]:
def fuzzy_search(data_df, difference_set, threshold):
    for e in difference_set:
        # Extract unique districts from data_df
        unique_districts = data_df['district'].unique()
        
        # Get top matches (max. 10)
        top_matches = process.extract(e, unique_districts, limit=10)
        
        print("----------------------------------------------------------")
        
        # Filter with threshold
        valid_matches = [match for match, score in top_matches if score >= threshold]
        
        # Render results
        if valid_matches:
            print(f"Geo: {e}")
            data_str = 'Data: ' + ' '.join([f"{i+1}. {match}" for i, match in enumerate(valid_matches)])
            print(data_str)
        else:
            print(f"Kein match für: {e}")
    

def fuzzy_match(row, district_set):
    # Extract region column from the row
    region = row['region']
    
    for district in district_set:
        if fuzz.partial_ratio(region, district) == 100:
            return True
    return False

def fuzzy_print(row, district_set):
    # Extract region column from the row
    region = row['region']
    for district in district_set:
        
        if fuzz.partial_ratio(region, district) == 100:
            print("-----")
            print(f"Region: {region}")
            print(f"District: {district}")
    return False

In [197]:
#lat, lon, zoom = 48.6616, 9.3601, 6.8 # BW

def set_geographical_values(selected_state):
    return STATE_GEO_VALUES.get(selected_state, STATE_GEO_VALUES['(Deutschland)'])

def create_state_map(state_df, selected_state, selected_building_type, selected_heat_source):
    lat, lon, zoom = set_geographical_values(selected_state)
    
    filtered_df = state_df.query(f"building_type == {selected_building_type} & heat_source == {selected_heat_source} & federal_state == '{selected_state}'")
    
    geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {"district": row.district, "district_type": row.district_type},
            "geometry": mapping(row.geometry)
        }
        for idx, row in filtered_df.iterrows()
    ]
    }
    
    fig = px.choropleth_mapbox(filtered_df, 
                            geojson=geojson, 
                            locations='district',
                            featureidkey="properties.district",
                            color='value',
                            color_continuous_scale='RdYlGn',
                            range_color=[0, 1],
                            mapbox_style="carto-darkmatter",
                            opacity=1,
                            #hover_data={'value': True},
                            hover_name=filtered_df['district'],
                            hover_data={'value': True} 
                           )

    fig.update_traces(hovertemplate='%{hovertext}<br>%{z:.1%}')
    fig.update_geos(fitbounds="locations")
    fig.update_layout(
        mapbox_zoom=zoom,
        mapbox_center={"lat": lat, "lon": lon},
        autosize=False,
        width=800, 
        height=800,
        coloraxis_showscale=False
        )
    
    return fig

In [339]:
def remove_cities(data_df):
    return data_df[~data_df['district'].str.endswith(', Kreisfreie Stadt')]

In [76]:
BUILDING_TYPE_MAPPING = {
        1: 'One- and Two-family Houses',
        6: 'Apartment Buildings (3-6)',
        9: 'Row Houses',
        11: 'Semi-detached Houses',
        38: 'Apartment Buildings: 7 and More Apartments',
        100: 'Total'
    }

HEAT_SOURCE_MAPPING = {
        0: 'Total',
        1: 'Air',
        2: 'Ground Probe',
        3: 'Ground Collector',
        4: 'Solar-Thermal Energy and Ice Storage'
    }

In [77]:
DISTRICT_DATA_URL = "https://opendata.ffe.de/api/od/v_opendata?id_opendata=eq.88"
DISTRICT_GEOJSON_URL = "https://raw.githubusercontent.com/isellsoap/deutschlandGeoJSON/main/4_kreise/2_hoch.geo.json"

## Process raw data

In [78]:
# Fetch data from API
def fetch_data(url):
    response = requests.get(url)
    if response.status_code == 200:
        return json.loads(response.text)
    else:
        return None

In [79]:
raw_data_json = fetch_data(DISTRICT_DATA_URL)

In [80]:
raw_data_df = pd.DataFrame(raw_data_json[0]['data']).dropna(axis=1, how='all')

In [81]:
drop_cols = ['id_opendata', 'id_region_type', 'region_type', 'id_region', 'year', 'internal_id']
raw_data_df.drop(drop_cols, axis=1, inplace=True)

In [82]:
raw_data_df.rename(columns={'region': 'district', 'internal_id_1': 'building_type', 'internal_id_2': 'heat_source'}, inplace=True)

## Process raw geojson

In [83]:
raw_geojson = fetch_data(DISTRICT_GEOJSON_URL)

In [84]:
# Initialize a dictionary to hold the features grouped by NAME_1
geojson_by_name_1 = defaultdict(list)

# Iterate over all features in the raw GeoJSON and group them by NAME_1
for feature in raw_geojson['features']:
    name_1 = feature['properties']['NAME_1']
    geojson_by_name_1[name_1].append(feature)

# Create a separate GeoJSON file for each unique NAME_1
output_dir = 'output_geojsons'
os.makedirs(output_dir, exist_ok=True)

for name_1, features in geojson_by_name_1.items():
    feature_collection = FeatureCollection(features)
    output_file_path = os.path.join(output_dir, f"{name_1.replace(' ', '_')}.geojson")
    
    with open(output_file_path, 'w') as f:
        json.dump(feature_collection, f)

print(f"GeoJSON created in the {output_dir} directory.")


GeoJSON created in the output_geojsons directory.


In [85]:
gpd_drop_cols = ['ID_0', 'ISO', 'NAME_0', 'ID_1', 'ID_2', 'NAME_2', 'ID_3', 'NL_NAME_3', 'VARNAME_3', 'ENGTYPE_3']

-----------------------------------------

## Baden-Württemberg

In [86]:
bw_geojson = gpd.read_file('output_geojsons/Baden-Württemberg.geojson')

### Process Geopandas Dataframe for Baden-Württemberg

In [132]:
# Create function for geopandas processing
def process_gpd_geojson(raw_gpd):
    """
    Geojson has to be read in with Geopandas first (gpd.read_file('filepath.geojson'))
    """
    gpd_drop_cols = ['ID_0', 'ISO', 'NAME_0', 'ID_1', 'ID_2', 'NAME_2', 'ID_3', 'NL_NAME_3', 'VARNAME_3', 'ENGTYPE_3']
    raw_gpd.drop(gpd_drop_cols, axis=1, inplace=True)
    renamed_cols = {
        'NAME_1': 'federal_state',
        'NAME_3': 'district',
        'TYPE_3': 'district_type'
    }
    raw_gpd.rename(columns=renamed_cols, inplace=True)
    
    # Defining the regex pattern
    pattern = re.compile(r' Städte')
    # Apply regex substitution
    raw_gpd['district'] = raw_gpd['district'].apply(lambda x: re.sub(pattern, '', x))
    
    # Sort
    raw_gpd = raw_gpd.sort_values(by=['district_type', 'district']).reset_index(drop=True)
    
    clean_gpd = raw_gpd
    
    return clean_gpd

In [87]:
gpd_drop_cols = ['ID_0', 'ISO', 'NAME_0', 'ID_1', 'ID_2', 'NAME_2', 'ID_3', 'NL_NAME_3', 'VARNAME_3', 'ENGTYPE_3']

In [88]:
bw_geojson.drop(gpd_drop_cols, axis=1, inplace=True)

In [89]:
renamed_cols = {
    'NAME_1': 'federal_state',
    'NAME_3': 'district',
    'TYPE_3': 'district_type'
}
bw_geojson.rename(columns=renamed_cols, inplace=True)

In [90]:
# Defining the regex pattern
pattern = re.compile(r' Städte')

In [91]:
# Apply regex substitution
bw_geojson['district'] = bw_geojson['district'].apply(lambda x: re.sub(pattern, '', x))

In [92]:
# Fix Heilbronn
bw_geojson.loc[bw_geojson['district'] == 'Heilbronn city', 'district'] = 'Heilbronn'

In [93]:
# Fix Freiburg (Freiburg im Breisgau)
bw_geojson.loc[bw_geojson['district'] == 'Freiburg', 'district'] = 'Freiburg im Breisgau'

In [94]:
# Sort
bw_geojson = bw_geojson.sort_values(by=['district_type', 'district']).reset_index(drop=True)

----------------------

#### ---DONE--- 

In [95]:
bw_geojson

,federal_state,district,district_type,geometry
0,Baden-Württemberg,Baden-Baden,Kreisfreie Städte,"POLYGON ((8.19391 48.83635, 8.19588 48.83392, ..."
1,Baden-Württemberg,Freiburg im Breisgau,Kreisfreie Städte,"POLYGON ((7.84910 48.05642, 7.84930 48.05272, ..."
2,Baden-Württemberg,Heidelberg,Kreisfreie Städte,"POLYGON ((8.58292 49.43313, 8.59429 49.43354, ..."
3,Baden-Württemberg,Heilbronn,Kreisfreie Städte,"POLYGON ((9.16618 49.20699, 9.16658 49.19575, ..."
4,Baden-Württemberg,Karlsruhe,Kreisfreie Städte,"POLYGON ((8.34080 49.08401, 8.34661 49.08050, ..."
5,Baden-Württemberg,Mannheim,Kreisfreie Städte,"POLYGON ((8.42372 49.57440, 8.42944 49.57460, ..."
6,Baden-Württemberg,Pforzheim,Kreisfreie Städte,"POLYGON ((8.70279 48.83761, 8.68022 48.83721, ..."
7,Baden-Württemberg,Stuttgart,Kreisfreie Städte,"POLYGON ((9.31032 48.77663, 9.29910 48.77277, ..."
8,Baden-Württemberg,Ulm,Kreisfreie Städte,"POLYGON ((10.00016 48.35505, 9.99456 48.35495,..."
9,Baden-Württemberg,Alb-Donau,Landkreise,"POLYGON ((9.58700 48.53665, 9.59255 48.54044, ..."


In [139]:
bw_geojson['district_type'].value_counts()

Landkreise           35
Kreisfreie Städte     9
Name: district_type, dtype: int64

GEODATA HAS 44 VALUES -> 35 LANDKREISE + 9 STÄDTE, KORREKT

-------------------------

#### Create raw_data subset

In [248]:
# CREATE FUNCTION FOR FUZZY MATCH
def fuzzy_match(raw_data_df, geo_df):
    # Create copy for editing
    data_df = raw_data_df.copy()
    geo = geo_df.copy()
    
    # Initialize match values
    data_df['match'] = False
    
    # Do the fuzzy match
    for df_i, df_r in data_df.iterrows():
        for geo_i, geo_r in geo.iterrows():
            if fuzz.partial_ratio(df_r['district'], geo_r['district']) == 100:
                geo.at[df_i, 'match'] = True
                break
    
    matched_data_df = data_df
    
    return matched_data_df

In [97]:
# Create copy for editing
bw_df = raw_data_df.copy()

In [98]:
# Initialize match values
bw_df['match'] = False

In [99]:
# Do the fuzzy match
for df_i, df_r in bw_df.iterrows():
    for geo_i, geo_r in bw_geojson.iterrows():
        if fuzz.partial_ratio(df_r['district'], geo_r['district']) == 100:
            bw_df.at[df_i, 'match'] = True
            break

In [100]:
# Set up for comparison
BWDATA = pd.DataFrame(bw_df[bw_df['match'] == True]['district'].unique())
BWDATA = BWDATA.rename(columns={0: 'data'})
BWDATA = BWDATA.sort_values(by='data').reset_index(drop=True)

In [101]:
BWGEO = pd.DataFrame(bw_geojson['district'])
BWGEO = BWGEO.rename(columns={'district': 'geo'})
BWGEO = BWGEO.sort_values(by='geo').reset_index(drop=True)

In [102]:
# Concat for comparison
pd.concat([BWDATA, BWGEO], axis=1)

,data,geo
0,Alb-Donau-Kreis,Alb-Donau
1,"Baden-Baden, Kreisfreie Stadt",Baden-Baden
2,Biberach,Biberach
3,Bodenseekreis,Bodensee
4,Breisgau-Hochschwarzwald,Breisgau-Hochschwarzwald
5,Böblingen,Böblingen
6,Calw,Calw
7,Emmendingen,Emmendingen
8,Enzkreis,Enz
9,Esslingen,Esslingen


Lindau(Bodensee), Neu-Ulm belong to Bayern -> remove

In [103]:
# Correct the flags
bw_df.loc[bw_df['district'] == 'Lindau (Bodensee)', 'match'] = False
bw_df.loc[bw_df['district'] == 'Neu-Ulm', 'match'] = False

In [104]:
# Create the BW subset
bw_df = bw_df.loc[bw_df['match'] == True]

In [105]:
# Remove flags
bw_df = bw_df.drop(columns='match')

In [106]:
# Save Baden-Württemberg Data
bw_df.to_csv("BW_data.csv")

### Merge Data with Geodata

In [110]:
bw_geo = bw_geojson.copy()

##### Start with "Kreisfreie Städte" 

In [111]:
bw_geo_cities = bw_geo[bw_geo['district_type'] == "Kreisfreie Städte"]

In [112]:
bw_geo_cities = bw_geo_cities.copy()
bw_geo_cities['district'] = bw_geo_cities['district'] + ', Kreisfreie Stadt'

In [113]:
bw_geo_cities

,federal_state,district,district_type,geometry
0,Baden-Württemberg,"Baden-Baden, Kreisfreie Stadt",Kreisfreie Städte,"POLYGON ((8.19391 48.83635, 8.19588 48.83392, ..."
1,Baden-Württemberg,"Freiburg im Breisgau, Kreisfreie Stadt",Kreisfreie Städte,"POLYGON ((7.84910 48.05642, 7.84930 48.05272, ..."
2,Baden-Württemberg,"Heidelberg, Kreisfreie Stadt",Kreisfreie Städte,"POLYGON ((8.58292 49.43313, 8.59429 49.43354, ..."
3,Baden-Württemberg,"Heilbronn, Kreisfreie Stadt",Kreisfreie Städte,"POLYGON ((9.16618 49.20699, 9.16658 49.19575, ..."
4,Baden-Württemberg,"Karlsruhe, Kreisfreie Stadt",Kreisfreie Städte,"POLYGON ((8.34080 49.08401, 8.34661 49.08050, ..."
5,Baden-Württemberg,"Mannheim, Kreisfreie Stadt",Kreisfreie Städte,"POLYGON ((8.42372 49.57440, 8.42944 49.57460, ..."
6,Baden-Württemberg,"Pforzheim, Kreisfreie Stadt",Kreisfreie Städte,"POLYGON ((8.70279 48.83761, 8.68022 48.83721, ..."
7,Baden-Württemberg,"Stuttgart, Kreisfreie Stadt",Kreisfreie Städte,"POLYGON ((9.31032 48.77663, 9.29910 48.77277, ..."
8,Baden-Württemberg,"Ulm, Kreisfreie Stadt",Kreisfreie Städte,"POLYGON ((10.00016 48.35505, 9.99456 48.35495,..."


In [114]:
# Merge city data with city geodata
bw_merged_cities = pd.merge(bw_df, bw_geo_cities, on='district')

In [115]:
bw_merged_cities

,district,building_type,heat_source,value,federal_state,district_type,geometry
0,"Stuttgart, Kreisfreie Stadt",100,0,0.5452,Baden-Württemberg,Kreisfreie Städte,"POLYGON ((9.31032 48.77663, 9.29910 48.77277, ..."
1,"Stuttgart, Kreisfreie Stadt",100,4,0.1697,Baden-Württemberg,Kreisfreie Städte,"POLYGON ((9.31032 48.77663, 9.29910 48.77277, ..."
2,"Stuttgart, Kreisfreie Stadt",100,2,0.3395,Baden-Württemberg,Kreisfreie Städte,"POLYGON ((9.31032 48.77663, 9.29910 48.77277, ..."
3,"Stuttgart, Kreisfreie Stadt",100,3,0.0944,Baden-Württemberg,Kreisfreie Städte,"POLYGON ((9.31032 48.77663, 9.29910 48.77277, ..."
4,"Stuttgart, Kreisfreie Stadt",100,1,0.4052,Baden-Württemberg,Kreisfreie Städte,"POLYGON ((9.31032 48.77663, 9.29910 48.77277, ..."
...,...,...,...,...,...,...,...
265,"Ulm, Kreisfreie Stadt",1,1,0.7955,Baden-Württemberg,Kreisfreie Städte,"POLYGON ((10.00016 48.35505, 9.99456 48.35495,..."
266,"Ulm, Kreisfreie Stadt",6,1,0.5913,Baden-Württemberg,Kreisfreie Städte,"POLYGON ((10.00016 48.35505, 9.99456 48.35495,..."
267,"Ulm, Kreisfreie Stadt",11,1,0.3488,Baden-Württemberg,Kreisfreie Städte,"POLYGON ((10.00016 48.35505, 9.99456 48.35495,..."
268,"Ulm, Kreisfreie Stadt",38,1,0.0718,Baden-Württemberg,Kreisfreie Städte,"POLYGON ((10.00016 48.35505, 9.99456 48.35495,..."


In [116]:
bw_merged_cities['district'].unique()

array(['Stuttgart, Kreisfreie Stadt', 'Heilbronn, Kreisfreie Stadt',
       'Baden-Baden, Kreisfreie Stadt', 'Karlsruhe, Kreisfreie Stadt',
       'Heidelberg, Kreisfreie Stadt', 'Mannheim, Kreisfreie Stadt',
       'Pforzheim, Kreisfreie Stadt',
       'Freiburg im Breisgau, Kreisfreie Stadt', 'Ulm, Kreisfreie Stadt'],
      dtype=object)

In [198]:
# Display all BW cities
fig = create_state_map(bw_merged_cities, 'Baden-Württemberg', 100, 1)
fig.show() # COMPLETE

#### Add Landkreise

In [119]:
# Remove already merged cities from data
bw_data_lk = bw_df[~bw_df['district'].str.endswith(', Kreisfreie Stadt')]

In [120]:
bw_data_lk.district.sort_values().unique()

array(['Alb-Donau-Kreis', 'Biberach', 'Bodenseekreis',
       'Breisgau-Hochschwarzwald', 'Böblingen', 'Calw', 'Emmendingen',
       'Enzkreis', 'Esslingen', 'Freudenstadt', 'Göppingen', 'Heidenheim',
       'Heilbronn', 'Hohenlohekreis', 'Karlsruhe', 'Konstanz',
       'Ludwigsburg', 'Lörrach', 'Main-Tauber-Kreis',
       'Neckar-Odenwald-Kreis', 'Ortenaukreis', 'Ostalbkreis', 'Rastatt',
       'Ravensburg', 'Rems-Murr-Kreis', 'Reutlingen',
       'Rhein-Neckar-Kreis', 'Rottweil', 'Schwarzwald-Baar-Kreis',
       'Schwäbisch Hall', 'Sigmaringen', 'Tuttlingen', 'Tübingen',
       'Waldshut', 'Zollernalbkreis'], dtype=object)

In [121]:
# Show geodata
bw_geo_lk = bw_geo[bw_geo['district_type'] != "Kreisfreie Städte"]

In [122]:
# Create dictionary for mismatches: 'geo':'data' -> convert to 'data'
bw_lk_match_dict = {
    'Alb-Donau': 'Alb-Donau-Kreis',
    'Bodensee': 'Bodenseekreis',
    'Enz': 'Enzkreis',
    'Hohenlohe': 'Hohenlohekreis',
    'Main-Tauber': 'Main-Tauber-Kreis'
}

In [123]:
bw_geo_lk

,federal_state,district,district_type,geometry
9,Baden-Württemberg,Alb-Donau,Landkreise,"POLYGON ((9.58700 48.53665, 9.59255 48.54044, ..."
10,Baden-Württemberg,Biberach,Landkreise,"POLYGON ((9.34670 48.24053, 9.36385 48.22967, ..."
11,Baden-Württemberg,Bodensee,Landkreise,"POLYGON ((9.41002 47.82195, 9.42666 47.82219, ..."
12,Baden-Württemberg,Breisgau-Hochschwarzwald,Landkreise,"POLYGON ((7.57809 48.11961, 7.58096 48.11911, ..."
13,Baden-Württemberg,Böblingen,Landkreise,"POLYGON ((8.92793 48.86775, 8.93925 48.86793, ..."
14,Baden-Württemberg,Calw,Landkreise,"POLYGON ((8.70279 48.83761, 8.70299 48.83383, ..."
15,Baden-Württemberg,Emmendingen,Landkreise,"POLYGON ((7.68154 48.25993, 7.70373 48.26050, ..."
16,Baden-Württemberg,Enz,Landkreise,"POLYGON ((8.87686 49.06552, 8.87763 49.05398, ..."
17,Baden-Württemberg,Esslingen,Landkreise,"POLYGON ((9.50889 48.75354, 9.51463 48.74993, ..."
18,Baden-Württemberg,Freudenstadt,Landkreise,"POLYGON ((8.22160 48.60462, 8.23282 48.60490, ..."


In [124]:
# Rename districts in geoset
bw_geo_lk = bw_geo_lk.copy()
bw_geo_lk['district'] = bw_geo_lk['district'].replace(bw_lk_match_dict)

In [125]:
bw_geo_lk

,federal_state,district,district_type,geometry
9,Baden-Württemberg,Alb-Donau-Kreis,Landkreise,"POLYGON ((9.58700 48.53665, 9.59255 48.54044, ..."
10,Baden-Württemberg,Biberach,Landkreise,"POLYGON ((9.34670 48.24053, 9.36385 48.22967, ..."
11,Baden-Württemberg,Bodenseekreis,Landkreise,"POLYGON ((9.41002 47.82195, 9.42666 47.82219, ..."
12,Baden-Württemberg,Breisgau-Hochschwarzwald,Landkreise,"POLYGON ((7.57809 48.11961, 7.58096 48.11911, ..."
13,Baden-Württemberg,Böblingen,Landkreise,"POLYGON ((8.92793 48.86775, 8.93925 48.86793, ..."
14,Baden-Württemberg,Calw,Landkreise,"POLYGON ((8.70279 48.83761, 8.70299 48.83383, ..."
15,Baden-Württemberg,Emmendingen,Landkreise,"POLYGON ((7.68154 48.25993, 7.70373 48.26050, ..."
16,Baden-Württemberg,Enzkreis,Landkreise,"POLYGON ((8.87686 49.06552, 8.87763 49.05398, ..."
17,Baden-Württemberg,Esslingen,Landkreise,"POLYGON ((9.50889 48.75354, 9.51463 48.74993, ..."
18,Baden-Württemberg,Freudenstadt,Landkreise,"POLYGON ((8.22160 48.60462, 8.23282 48.60490, ..."


In [126]:
bw_merged_lk = pd.merge(bw_data_lk, bw_geo_lk, on='district')

In [127]:
bw_merged_lk

,district,building_type,heat_source,value,federal_state,district_type,geometry
0,Böblingen,100,0,0.6385,Baden-Württemberg,Landkreise,"POLYGON ((8.92793 48.86775, 8.93925 48.86793, ..."
1,Böblingen,11,3,0.0571,Baden-Württemberg,Landkreise,"POLYGON ((8.92793 48.86775, 8.93925 48.86793, ..."
2,Böblingen,6,3,0.0893,Baden-Württemberg,Landkreise,"POLYGON ((8.92793 48.86775, 8.93925 48.86793, ..."
3,Böblingen,38,3,0.0422,Baden-Württemberg,Landkreise,"POLYGON ((8.92793 48.86775, 8.93925 48.86793, ..."
4,Böblingen,1,1,0.7280,Baden-Württemberg,Landkreise,"POLYGON ((8.92793 48.86775, 8.93925 48.86793, ..."
...,...,...,...,...,...,...,...
1045,Sigmaringen,6,1,0.6734,Baden-Württemberg,Landkreise,"POLYGON ((9.34670 48.24053, 9.33546 48.24036, ..."
1046,Sigmaringen,9,1,0.3340,Baden-Württemberg,Landkreise,"POLYGON ((9.34670 48.24053, 9.33546 48.24036, ..."
1047,Sigmaringen,38,1,0.1566,Baden-Württemberg,Landkreise,"POLYGON ((9.34670 48.24053, 9.33546 48.24036, ..."
1048,Sigmaringen,1,1,0.8568,Baden-Württemberg,Landkreise,"POLYGON ((9.34670 48.24053, 9.33546 48.24036, ..."


In [231]:
# Display all BW Landkreise
fig = create_state_map(bw_merged_lk, 'Baden-Württemberg', 100, 1)
fig.show() # COMPLETE

In [54]:
# Merge Cities with Landkreise
bw_full = pd.concat([bw_merged_cities, bw_merged_lk], axis=0)

In [235]:
# Display full map
fig = create_state_map(bw_full, 'Baden-Württemberg', 100, 0)
fig.show()

--------------------------

<center><b>BADEN-WÜRTTEMBERG MAP IS COMPLETE</b></center>

-------------------

In [129]:
# Export data
# bw_full.to_csv("bw_data_geo.csv")

### BAYERN

In [430]:
# Create copy for editing
by_df = raw_data_df.copy()

####  Process geodata

In [431]:
# Read in geojson -> geopandas
by_geojson = gpd.read_file('output_geojsons/Bayern.geojson')

In [471]:
by_geojson

,federal_state,district,district_type,geometry
0,Bayern,Amberg,Kreisfreie Städte,"POLYGON ((11.88697 49.49106, 11.89257 49.48726..."
1,Bayern,Ansbach,Kreisfreie Städte,"POLYGON ((10.59538 49.33940, 10.60123 49.33205..."
2,Bayern,Aschaffenburg,Kreisfreie Städte,"POLYGON ((9.25161 49.93389, 9.23453 49.93756, ..."
3,Bayern,Augsburg,Kreisfreie Städte,"POLYGON ((10.89148 48.45751, 10.89706 48.45746..."
4,Bayern,Bamberg,Kreisfreie Städte,"POLYGON ((10.94377 49.90851, 10.94961 49.90117..."
5,Bayern,Bayreuth,Kreisfreie Städte,"POLYGON ((11.56640 49.97448, 11.56642 49.97077..."
6,Bayern,Coburg,Kreisfreie Städte,"POLYGON ((10.96433 50.29026, 10.97588 50.28289..."
7,Bayern,Erlangen,Kreisfreie Städte,"POLYGON ((10.94085 49.53735, 10.94074 49.55227..."
8,Bayern,Fürth,Kreisfreie Städte,"POLYGON ((10.99837 49.53417, 11.00413 49.53049..."
9,Bayern,Hof,Kreisfreie Städte,"POLYGON ((11.89541 50.35578, 11.89523 50.35204..."


In [432]:
# Process
by_geojson = process_gpd_geojson(by_geojson)

In [433]:
# Check
by_geojson['district_type'].value_counts() # KORREKT

Landkreise           71
Kreisfreie Städte    25
Name: district_type, dtype: int64

In [434]:
by_geojson

,federal_state,district,district_type,geometry
0,Bayern,Amberg,Kreisfreie Städte,"POLYGON ((11.88697 49.49106, 11.89257 49.48726..."
1,Bayern,Ansbach,Kreisfreie Städte,"POLYGON ((10.59538 49.33940, 10.60123 49.33205..."
2,Bayern,Aschaffenburg,Kreisfreie Städte,"POLYGON ((9.25161 49.93389, 9.23453 49.93756, ..."
3,Bayern,Augsburg,Kreisfreie Städte,"POLYGON ((10.89148 48.45751, 10.89706 48.45746..."
4,Bayern,Bamberg,Kreisfreie Städte,"POLYGON ((10.94377 49.90851, 10.94961 49.90117..."
5,Bayern,Bayreuth,Kreisfreie Städte,"POLYGON ((11.56640 49.97448, 11.56642 49.97077..."
6,Bayern,Coburg,Kreisfreie Städte,"POLYGON ((10.96433 50.29026, 10.97588 50.28289..."
7,Bayern,Erlangen,Kreisfreie Städte,"POLYGON ((10.94085 49.53735, 10.94074 49.55227..."
8,Bayern,Fürth,Kreisfreie Städte,"POLYGON ((10.99837 49.53417, 11.00413 49.53049..."
9,Bayern,Hof,Kreisfreie Städte,"POLYGON ((11.89541 50.35578, 11.89523 50.35204..."


In [435]:
by_geo = by_geojson.copy()
by_geo_cities = by_geo[by_geo['district_type'] == 'Kreisfreie Städte'] 

In [436]:
# Create match dictionary
by_match_dict = {
    'Munich': 'München',
    'Nuremberg': 'Nürnberg',
    'Kempten': 'Kempten (Allgäu)', # identified later
    'Weiden': 'Weiden i.d. OPf.' # identified later
    
}
# INTEGRATED IDENTIFIED MISSING MATCHES (Kempten, Weiden)

In [437]:
# Renaming
by_geo_cities['district'] = by_geo_cities['district'].replace(by_match_dict)

/home/th/anaconda3/lib/python3.10/site-packages/geopandas/geodataframe.py:1538: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [438]:
by_geo_cities = by_geo_cities.copy()
by_geo_cities['district'] = by_geo_cities['district'] + ', Kreisfreie Stadt'

In [439]:
by_geo_cities

,federal_state,district,district_type,geometry
0,Bayern,"Amberg, Kreisfreie Stadt",Kreisfreie Städte,"POLYGON ((11.88697 49.49106, 11.89257 49.48726..."
1,Bayern,"Ansbach, Kreisfreie Stadt",Kreisfreie Städte,"POLYGON ((10.59538 49.33940, 10.60123 49.33205..."
2,Bayern,"Aschaffenburg, Kreisfreie Stadt",Kreisfreie Städte,"POLYGON ((9.25161 49.93389, 9.23453 49.93756, ..."
3,Bayern,"Augsburg, Kreisfreie Stadt",Kreisfreie Städte,"POLYGON ((10.89148 48.45751, 10.89706 48.45746..."
4,Bayern,"Bamberg, Kreisfreie Stadt",Kreisfreie Städte,"POLYGON ((10.94377 49.90851, 10.94961 49.90117..."
5,Bayern,"Bayreuth, Kreisfreie Stadt",Kreisfreie Städte,"POLYGON ((11.56640 49.97448, 11.56642 49.97077..."
6,Bayern,"Coburg, Kreisfreie Stadt",Kreisfreie Städte,"POLYGON ((10.96433 50.29026, 10.97588 50.28289..."
7,Bayern,"Erlangen, Kreisfreie Stadt",Kreisfreie Städte,"POLYGON ((10.94085 49.53735, 10.94074 49.55227..."
8,Bayern,"Fürth, Kreisfreie Stadt",Kreisfreie Städte,"POLYGON ((10.99837 49.53417, 11.00413 49.53049..."
9,Bayern,"Hof, Kreisfreie Stadt",Kreisfreie Städte,"POLYGON ((11.89541 50.35578, 11.89523 50.35204..."


In [440]:
by_merged_cities = pd.merge(by_df, by_geo_cities, on='district')

In [441]:
# Display all Bayern cities
fig = create_state_map(by_merged_cities, 'Bayern', 100, 1)
fig.show()

In [442]:
# Remove already merged cities from data
by_data_lk = remove_cities(by_df)

In [443]:
# Show geodata
by_geo_lk = by_geo[by_geo['district_type'] != "Kreisfreie Städte"]

In [444]:
by_geo_lk.shape # Geodaten for 71 "Landkreise"

(71, 4)

In [445]:
by_df

,district,building_type,heat_source,value
0,"Flensburg, Kreisfreie Stadt",100,0,0.6492
1,"Flensburg, Kreisfreie Stadt",100,4,0.3281
2,"Flensburg, Kreisfreie Stadt",100,2,0.4203
3,"Flensburg, Kreisfreie Stadt",100,3,0.2073
4,"Flensburg, Kreisfreie Stadt",100,1,0.5565
...,...,...,...,...
12055,Ludwigslust-Parchim,100,4,0.5854
12056,Ludwigslust-Parchim,100,2,0.6517
12057,Ludwigslust-Parchim,100,3,0.4252
12058,Ludwigslust-Parchim,100,1,0.8306


In [446]:
# Merge data with LK
by_merged_lk = pd.merge(by_df, by_geo_lk, on='district')

In [447]:
# Map with all "Landkreise"
fig = create_state_map(by_merged_lk, "Bayern", 100, 0)
fig.show()

In [448]:
1860 / 30 # = 62 -> 9 Landkreise missing (71 in total)

62.0

In [449]:
# Identify missing Landkreise
set_by_geo_lk = set(by_geo_lk['district'])
set_by_merged_lk = set(by_merged_lk['district'])

In [450]:
set_diff_BY = set_by_geo_lk - set_by_merged_lk
set_diff_BY # Namen aus GEO welche keinen Match in DATA gefunden haben

{'Dillingen',
 'Landsberg',
 'Lindau',
 'Mühldorf',
 'Neumarkt',
 'Neustadt',
 'Neustadt-Bad Windsheim',
 'Pfaffenhofen',
 'Wunsiedel'}

In [451]:
fuzzy_search(by_df, set_diff_by, 80)

----------------------------------------------------------
Geo: Neumarkt
Data: 1. Neumarkt i.d. OPf.
----------------------------------------------------------
Geo: Landsberg
Data: 1. Landsberg am Lech
----------------------------------------------------------
Geo: Neustadt-Bad Windsheim
Data: 1. Bad Kreuznach 2. Neustadt an der Weinstraße, Kreisfreie Stadt 3. Bad Dürkheim 4. Neustadt a.d. Aisch-Bad Windsheim 5. Bad Kissingen
----------------------------------------------------------
Geo: Neustadt
Data: 1. Neustadt an der Weinstraße, Kreisfreie Stadt 2. Neustadt a.d. Waldnaab 3. Neustadt a.d. Aisch-Bad Windsheim
----------------------------------------------------------
Geo: Pfaffenhofen
Data: 1. Pfaffenhofen a.d. Ilm 2. Hof
----------------------------------------------------------
Geo: Lindau
Data: 1. Lindau (Bodensee)
----------------------------------------------------------
Geo: Wunsiedel
Data: 1. Wunsiedel i. Fichtelgebirge
--------------------------------------------------------

In [452]:
# Rename districts in geodata to match correct corresponding districts in the heat pump dataset
by_geo_lk.loc[by_geo_lk['district'] == 'Neumarkt', 'district'] = 'Neumarkt i.d. OPf.'
by_geo_lk.loc[by_geo_lk['district'] == 'Landsberg', 'district'] = 'Landsberg am Lech'
by_geo_lk.loc[by_geo_lk['district'] == 'Neustadt-Bad Windsheim', 'district'] = 'Neustadt a.d. Aisch-Bad Windsheim'
by_geo_lk.loc[by_geo_lk['district'] == 'Neustadt', 'district'] = 'Neustadt a.d. Waldnaab'
by_geo_lk.loc[by_geo_lk['district'] == 'Pfaffenhofen', 'district'] = 'Pfaffenhofen a.d. Ilm'
by_geo_lk.loc[by_geo_lk['district'] == 'Lindau', 'district'] = 'Lindau (Bodensee)'
by_geo_lk.loc[by_geo_lk['district'] == 'Wunsiedel', 'district'] = 'Wunsiedel i. Fichtelgebirge'
by_geo_lk.loc[by_geo_lk['district'] == 'Mühldorf', 'district'] = 'Mühldorf a. Inn'
by_geo_lk.loc[by_geo_lk['district'] == 'Dillingen', 'district'] = 'Dillingen a.d. Donau'

In [453]:
# Update merged_lk with full geo lk data
by_merged_lk = pd.merge(by_df, by_geo_lk, on='district')

In [454]:
# Map with all "Landkreise", complete
fig = create_state_map(by_merged_lk, "Bayern", 100, 0)
fig.show()

In [455]:
# Merge with cities
by_full = pd.concat([by_merged_cities, by_merged_lk])

In [540]:
# Draw full map for Bayern
fig = create_state_map(by_full, "Bayern", 100, 1)
fig.show()

-> still missing: Kempten (Allgäu) [FIXED], Weiden in der Oberpfalz [FIXED]

In [361]:
by_df[by_df['district'].str.startswith("Kempten")][['district']].head(1)

,district
9180,"Kempten (Allgäu), Kreisfreie Stadt"


In [408]:
by_df[by_df['district'].str.startswith("Weiden")][['district']].head(1)

,district
7770,"Weiden i.d. OPf., Kreisfreie Stadt"


In [459]:
# Export Bayern data
# by_full.to_csv("by_full_data.csv")

------------------------------------------------------------------------

<center><b>BAYERN MAP IS COMPLETE</b></center>

-----------------------

### BRANDENBURG 

In [501]:
# Create DATA copy for editing
bb_df = raw_data_df.copy()

In [502]:
# Read in geojson -> geopandas
bb_geojson = gpd.read_file(f'output_geojsons/Brandenburg.geojson')

# Process
bb_geojson = process_gpd_geojson(bb_geojson)

# Check
bb_geojson['district_type'].value_counts() # CORRECT

Landkreise           14
Kreisfreie Städte     4
Name: district_type, dtype: int64

-----------------------------------------------

In [503]:
bb_geo = bb_geojson.copy()
bb_geo_cities = bb_geo[bb_geo['district_type'] == 'Kreisfreie Städte']
bb_geo_lk = bb_geo[bb_geo['district_type'] != 'Kreisfreie Städte'] 

In [504]:
bb_geo_cities[['district']]

,district
0,Brandenburg an der Havel
1,Cottbus
2,Frankfurt am Oder
3,Potsdam


In [505]:
bb_cities_set = set(bb_geo_cities['district'])
bb_lk_set = set(bb_geo_lk['district'])

In [506]:
fuzzy_search(bb_df, bb_cities_set, 80)

----------------------------------------------------------
Geo: Frankfurt am Oder
Data: 1. Frankfurt am Main, Kreisfreie Stadt 2. Offenbach am Main, Kreisfreie Stadt 3. Ludwigshafen am Rhein, Kreisfreie Stadt 4. Frankfurt (Oder), Kreisfreie Stadt 5. Oder-Spree
----------------------------------------------------------
Geo: Cottbus
Data: 1. Cottbus, Kreisfreie Stadt
----------------------------------------------------------
Geo: Potsdam
Data: 1. Potsdam, Kreisfreie Stadt 2. Potsdam-Mittelmark
----------------------------------------------------------
Geo: Brandenburg an der Havel
Data: 1. Brandenburg an der Havel, Kreisfreie Stadt 2. Mülheim an der Ruhr, Kreisfreie Stadt 3. Landau in der Pfalz, Kreisfreie Stadt 4. Neustadt an der Weinstraße, Kreisfreie Stadt


In [507]:
# Create cities match dictionary for Brandenburg
bb_cities_match_dict = {
    'Frankfurt am Oder': 'Frankfurt (Oder)',
    'Cottbus': 'Cottbus, Kreisfreie Stadt',
    'Potsdam': 'Potsdam, Kreisfreie Stadt',
    'Brandenburg an der Havel': 'Brandenburg an der Havel, Kreisfreie Stadt'
}

In [508]:
# Rename cities w/ cities match dict
bb_geo.loc[:, 'district'] = bb_geo['district'].replace(bb_cities_match_dict)

In [510]:
fuzzy_search(bb_df, bb_lk_set, 80) # Everything's fine -> bb_geo is ready for full merge

----------------------------------------------------------
Geo: Teltow-Fläming
Data: 1. Teltow-Fläming
----------------------------------------------------------
Geo: Prignitz
Data: 1. Prignitz 2. Ostprignitz-Ruppin
----------------------------------------------------------
Geo: Potsdam-Mittelmark
Data: 1. Potsdam-Mittelmark
----------------------------------------------------------
Geo: Dahme-Spreewald
Data: 1. Dahme-Spreewald
----------------------------------------------------------
Geo: Elbe-Elster
Data: 1. Elbe-Elster
----------------------------------------------------------
Geo: Spree-Neiße
Data: 1. Spree-Neiße
----------------------------------------------------------
Geo: Oder-Spree
Data: 1. Oder-Spree 2. Frankfurt (Oder), Kreisfreie Stadt
----------------------------------------------------------
Geo: Oberspreewald-Lausitz
Data: 1. Oberspreewald-Lausitz
----------------------------------------------------------
Geo: Oberhavel
Data: 1. Oberhavel
-------------------------------

In [512]:
bb_full = pd.merge(bb_df, bb_geo, on='district')

In [534]:
fig = create_state_map(bb_full, 'Brandenburg', 100, 0)
fig.show()

-> TODO: DISPLAY TOGETHER WITH BERLIN

In [533]:
# Export Brandenburg data
# bb_full.to_csv("bb_full_data.csv")

------------------

<center><b>BRANDENBURG MAP IS COMPLETE</b></center>

------------------------------------

### Bremen 

In [542]:
# Create DATA copy for editing
hb_df = raw_data_df.copy()

In [543]:
# Read in geojson -> geopandas
hb_geojson = gpd.read_file(f'output_geojsons/Bremen.geojson')

# Process
hb_geojson = process_gpd_geojson(hb_geojson)

# Check
hb_geojson['district_type'].value_counts() # BREMERHAVEN IS MISSING

Kreisfreie Städte    1
Name: district_type, dtype: int64

In [546]:
hb_geojson

,federal_state,district,district_type,geometry
0,Bremen,Bremen,Kreisfreie Städte,"POLYGON ((8.98545 53.12822, 8.97316 53.12799, ..."


In [550]:
hb_df[hb_df['district'].str.startswith('Breme')]['district'].unique()

array(['Bremen, Kreisfreie Stadt', 'Bremerhaven, Kreisfreie Stadt'],
      dtype=object)

TODO: Bremerhaven GEO is missing, get a different source 

### Hamburg

In [551]:
# Create DATA copy for editing
hh_df = raw_data_df.copy()

In [553]:
# Read in geojson -> geopandas
hh_geojson = gpd.read_file(f'output_geojsons/Hamburg.geojson')

# Process
hh_geojson = process_gpd_geojson(hh_geojson)

# Check
hh_geojson['district_type'].value_counts() # CORRECT

Kreisfreie Städte    1
Name: district_type, dtype: int64

In [554]:
hh_geo = hh_geojson.copy()
hh_geo

,federal_state,district,district_type,geometry
0,Hamburg,Hamburg,Kreisfreie Städte,"POLYGON ((10.07162 53.71823, 10.07150 53.72192..."


In [558]:
hh_full = pd.merge(hh_df, hh_geo, on='district')

In [562]:
fig = create_state_map(hh_full, 'Hamburg', 100, 0)
fig.show()

In [564]:
# hh_full.to_csv("hh_full_data.csv")

------------------------------

<center><b>HAMBURG MAP IS COMPLETE</b></center>

------------------------------

### Hessen

In [565]:
# Create DATA copy for editing
he_df = raw_data_df.copy()

In [566]:
# Read in geojson -> geopandas
he_geojson = gpd.read_file(f'output_geojsons/Hessen.geojson')

# Process
he_geojson = process_gpd_geojson(he_geojson)

# Check
he_geojson['district_type'].value_counts() # should be 21 Landkreise + 5 Kreisfreie Städte

Landkreise           20
Kreisfreie Städte     6
Name: district_type, dtype: int64